
# [SQL 실습 #04] MySQL 데이터 조회하기 — SELECT 기초

> **학생용 실습 노트북 - TODO 완성형**

지난 시간에는 `INSERT INTO`를 이용해 `students` 테이블에 데이터를 입력했습니다.  
이번 시간에는 저장된 데이터를 **꺼내 보는 방법**, 즉 `SELECT`를 배웁니다.

---

## 오늘의 학습 목표

실습을 마치면 다음을 할 수 있어야 합니다.

1. `SELECT` 기본 문법을 설명할 수 있다.
2. 전체 데이터를 조회할 수 있다.
3. 원하는 컬럼만 조회할 수 있다.
4. 컬럼 이름에 별명(`AS`)을 붙일 수 있다.
5. `WHERE`로 조건을 걸어 조회할 수 있다.
6. 비교 연산자를 사용할 수 있다.
7. `AND`, `OR`, `IN`으로 여러 조건을 처리할 수 있다.
8. `LIKE`로 특정 글자가 포함된 데이터를 찾을 수 있다.
9. `ORDER BY`로 정렬할 수 있다.
10. `LIMIT`으로 조회 개수를 제한할 수 있다.
11. `DISTINCT`로 중복 값을 제거할 수 있다.
12. `COUNT(*)`로 행 개수를 셀 수 있다.

---

## 오늘의 핵심

> **SELECT는 테이블에 저장된 데이터를 꺼내 보는 SQL 명령어입니다.**

쉽게 말하면,

> `INSERT`가 서류철에 자료를 넣는 일이라면,  
> `SELECT`는 서류철에서 필요한 자료를 찾아 꺼내 보는 일입니다.


---

## ✅ 이번 버전의 자동점검 기능

각 TODO 코드 셀 바로 아래에 **자동점검 셀**이 있습니다.

1. TODO를 직접 완성합니다.
2. TODO 코드 셀을 실행합니다.
3. 바로 아래 `자동점검` 셀을 실행합니다.
4. 결과가 맞으면 `✅ 통과`, 다르면 `❌ 재확인`이 표시됩니다.
5. 마지막 **전체 TODO 자동점검 결과**에서 25개 TODO의 진행률을 확인할 수 있습니다.

자동점검은 SQL 문장을 단순 문자열로 비교하지 않고 **실제 조회 결과를 비교**합니다.  
따라서 같은 결과를 만드는 올바른 SQL이라면 표현 방식이 조금 달라도 통과할 수 있습니다.



# 1. 실습 환경 준비

코랩 런타임이 초기화되면 MySQL과 이전 데이터가 사라질 수 있습니다.

이번 노트북에서는 아래 작업을 자동으로 준비합니다.

- MySQL 설치 및 실행
- `school` 데이터베이스 생성
- `students` 테이블 생성
- 조회 실습용 데이터 7개 입력

> 아래 셀은 수정하지 말고 실행하세요.


In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start

print("✅ MySQL 서버 준비 완료")



# 2. 조회 실습용 데이터 준비

이번 실습에서는 다음 7개의 데이터를 사용합니다.

| name | grade | class_name |
|---|---:|---|
| 김부장 | 3 | 컴퓨터과 |
| 혼이 | 1 | 영혼반 |
| 라즈베리 | 2 | 임베디드반 |
| 리눅스 | 3 | 서버반 |
| 파이썬 | 1 | 프로그래밍반 |
| 김코딩 | 1 | 컴퓨터과 |
| 데이터왕 | 2 | 데이터베이스반 |

테이블 구조는 기존 실습과 동일합니다.

```text
id
name
grade
class_name
created_at
```

아래 셀은 실습 데이터를 초기화하고 다시 넣습니다.


In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name)
VALUES
('김부장', 3, '컴퓨터과'),
('혼이', 1, '영혼반'),
('라즈베리', 2, '임베디드반'),
('리눅스', 3, '서버반'),
('파이썬', 1, '프로그래밍반'),
('김코딩', 1, '컴퓨터과'),
('데이터왕', 2, '데이터베이스반');
SQL

echo "✅ school.students 조회 실습 데이터 준비 완료"



# 3. SQL 실행 도우미 함수

아래 함수는 학생이 작성한 SQL을 실행해 줍니다.

- SQL이 비어 있으면 실행하지 않음
- `TODO`가 남아 있으면 실행하지 않음
- MySQL 오류가 나면 오류 메시지를 표시

> 이 셀은 수정하지 마세요.


In [ ]:
import subprocess
import textwrap
import base64
import re

LAST_SQL = ""
LAST_SQL_OK = False
TODO_STATUS = {}

def _execute_mysql(sql):
    """SQL을 mysql batch 모드로 실행하고 (성공여부, stdout, stderr)를 반환합니다."""
    result = subprocess.run(
        ["sudo", "mysql", "--batch", "--raw"],
        input=sql,
        text=True,
        capture_output=True
    )
    return result.returncode == 0, result.stdout.strip(), result.stderr.strip()

def run_sql(sql):
    """학생 SQL 실행용 함수"""
    global LAST_SQL, LAST_SQL_OK

    sql = textwrap.dedent(sql).strip()
    LAST_SQL = sql
    LAST_SQL_OK = False

    executable_lines = [
        line for line in sql.splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ]
    executable_sql = "\n".join(executable_lines).strip()

    if not executable_sql:
        print("⚠️ 아직 SQL이 작성되지 않았습니다.")
        return False

    if "TODO" in executable_sql:
        print("⚠️ SQL 안에 TODO가 남아 있습니다. 먼저 완성하세요.")
        return False

    ok, out, err = _execute_mysql(sql)

    if out:
        print(out)

    if not ok:
        print("❌ MySQL 오류가 발생했습니다.")
        print(err)
        LAST_SQL_OK = False
        return False

    LAST_SQL_OK = True
    print("✅ SQL 실행 완료")
    return True


# ---------------------------------------------------------------
# 자동점검용 기대 SQL
# 정답 SQL을 학생 화면에 바로 노출하지 않도록 Base64로 저장합니다.
# 결과 비교 방식이므로 문법이 조금 달라도 결과가 같으면 통과할 수 있습니다.
# ---------------------------------------------------------------
_EXPECTED_B64 = {
    1: 'VVNFIHNjaG9vbDsKU0hPVyBUQUJMRVM7CkRFU0Mgc3R1ZGVudHM7',
    2: 'VVNFIHNjaG9vbDsKU0VMRUNUIG5hbWUgRlJPTSBzdHVkZW50czs=',
    3: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50czs=',
    4: 'VVNFIHNjaG9vbDsKU0VMRUNUIG5hbWUsIGdyYWRlLCBjbGFzc19uYW1lIEZST00gc3R1ZGVudHM7',
    5: 'VVNFIHNjaG9vbDsKU0VMRUNUIG5hbWUgQVMg7J2066aELCBncmFkZSBBUyDtlZnrhYQsIGNsYXNzX25hbWUgQVMg7ZWZ6rO8IEZST00gc3R1ZGVudHM7',
    6: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA9IDM7',
    7: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lID0gJ+y7tO2TqO2EsOqzvCc7',
    8: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA+PSAyOw==',
    9: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSAhPSAxOw==',
    10: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA9IDMgQU5EIGNsYXNzX25hbWUgPSAn7Lu07ZOo7YSw6rO8Jzs=',
    11: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA9IDEgT1IgZ3JhZGUgPSAyOw==',
    12: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBJTiAoMSwgMik7',
    13: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIElOICgn7Lu07ZOo7YSw6rO8JywgJ+yEnOuyhOuwmCcpOw==',
    14: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lIExJS0UgJ+q5gCUnOw==',
    15: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBPUkRFUiBCWSBncmFkZSBBU0M7',
    16: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBPUkRFUiBCWSBncmFkZSBERVNDOw==',
    17: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBPUkRFUiBCWSBncmFkZSBBU0MsIG5hbWUgQVNDOw==',
    18: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBMSU1JVCAzOw==',
    19: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBPUkRFUiBCWSBpZCBERVNDIExJTUlUIDM7',
    20: 'VVNFIHNjaG9vbDsKU0VMRUNUIERJU1RJTkNUIGNsYXNzX25hbWUgRlJPTSBzdHVkZW50czs=',
    21: 'VVNFIHNjaG9vbDsKU0VMRUNUIENPVU5UKCopIEZST00gc3R1ZGVudHM7',
    22: 'VVNFIHNjaG9vbDsKU0VMRUNUIENPVU5UKCopIEZST00gc3R1ZGVudHMgV0hFUkUgZ3JhZGUgPSAxOw==',
    23: 'VVNFIHNjaG9vbDsKU0VMRUNUIG5hbWUsIGdyYWRlIEZST00gc3R1ZGVudHMgV0hFUkUgZ3JhZGUgPj0gMiBPUkRFUiBCWSBncmFkZSBERVNDLCBuYW1lIEFTQzs=',
    24: 'VVNFIHNjaG9vbDsKU0VMRUNUIG5hbWUsIGdyYWRlLCBjbGFzc19uYW1lIEZST00gc3R1ZGVudHMgV0hFUkUgY2xhc3NfbmFtZSBJTiAoJ+y7tO2TqO2EsOqzvCcsICfshJzrsoTrsJgnKSBPUkRFUiBCWSBuYW1lIEFTQzs=',
    25: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lIExJS0UgJyXquYAlJzs=',
}

_ORDER_SENSITIVE = {15, 16, 17, 18, 19, 23, 24}

def _decode_expected(todo_id):
    return base64.b64decode(_EXPECTED_B64[todo_id]).decode("utf-8")

def _normalize_output(text, todo_id, ordered=False):
    """
    mysql --batch 결과를 비교 가능한 형태로 정규화합니다.
    - 공백/빈 줄 정리
    - TODO 1처럼 여러 결과가 연속되는 경우는 출력 전체를 그대로 비교
    - ORDER BY가 핵심인 TODO는 행 순서를 유지
    - 그 외 조회는 헤더를 유지하고 데이터 행만 정렬하여
      동등한 결과의 SQL을 허용
    """
    lines = [line.rstrip() for line in text.strip().splitlines() if line.strip()]
    if not lines:
        return []

    if todo_id == 1:
        return lines

    header = lines[0]
    rows = lines[1:]
    if not ordered:
        rows = sorted(rows)
    return [header] + rows

def check_todo(todo_id, sql, current_todo_id=None):
    """
    학생이 해당 TODO 셀을 실제로 실행했는지와 결과가 맞는지 자동 확인합니다.
    """
    global TODO_STATUS

    print(f"🔎 TODO {todo_id} 자동점검")

    if current_todo_id != todo_id:
        TODO_STATUS[todo_id] = False
        print("❌ 바로 위 TODO 코드 셀을 먼저 실행하세요.")
        return False

    student_sql = textwrap.dedent(sql).strip()

    real_lines = [
        line for line in student_sql.splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ]
    real_sql = "\n".join(real_lines).strip()

    if not real_sql:
        TODO_STATUS[todo_id] = False
        print("❌ SQL이 비어 있습니다.")
        return False

    if "TODO" in real_sql:
        TODO_STATUS[todo_id] = False
        print("❌ TODO가 아직 남아 있습니다.")
        return False

    ok_student, out_student, err_student = _execute_mysql(student_sql)
    if not ok_student:
        TODO_STATUS[todo_id] = False
        print("❌ SQL 실행 오류가 있습니다.")
        print(err_student)
        return False

    expected_sql = _decode_expected(todo_id)
    ok_expected, out_expected, err_expected = _execute_mysql(expected_sql)
    if not ok_expected:
        TODO_STATUS[todo_id] = False
        print("⚠️ 자동점검 기준 실행 중 오류가 발생했습니다. 교사에게 알려주세요.")
        return False

    ordered = todo_id in _ORDER_SENSITIVE

    student_norm = _normalize_output(out_student, todo_id, ordered=ordered)
    expected_norm = _normalize_output(out_expected, todo_id, ordered=ordered)

    passed = (student_norm == expected_norm)
    TODO_STATUS[todo_id] = passed

    if passed:
        print("✅ 통과: 요구한 결과가 정확합니다.")
    else:
        print("❌ 재확인: 실행은 되었지만 요구한 결과와 다릅니다.")
        print("   → SELECT 컬럼, 조건, 정렬 방향, LIMIT 등을 다시 확인하세요.")

    return passed


def show_todo_progress(total=25):
    """TODO 전체 진행률 출력"""
    done = sum(1 for i in range(1, total + 1) if TODO_STATUS.get(i) is True)
    failed = sum(1 for i in range(1, total + 1) if TODO_STATUS.get(i) is False)
    not_checked = total - done - failed

    print("=" * 45)
    print("📊 TODO 자동점검 진행 현황")
    print("=" * 45)

    for start in range(1, total + 1, 5):
        end = min(start + 4, total)
        parts = []
        for i in range(start, end + 1):
            state = TODO_STATUS.get(i)
            mark = "✅" if state is True else "❌" if state is False else "⬜"
            parts.append(f"{i}:{mark}")
        print("   ".join(parts))

    print("-" * 45)
    print(f"통과       : {done}/{total}")
    print(f"재확인 필요: {failed}/{total}")
    print(f"미점검     : {not_checked}/{total}")
    print(f"진행률     : {done / total * 100:.1f}%")

    if done == total:
        print("\n🎉 모든 TODO를 정확하게 완료했습니다!")
    else:
        print("\n💡 ⬜ 또는 ❌인 TODO를 다시 확인하세요.")



# 4. 현재 데이터베이스와 테이블 확인

조회하기 전에 작업 대상이 맞는지 확인합니다.

## TODO 1

다음을 수행하세요.

1. `school` 데이터베이스 선택
2. 테이블 목록 확인
3. `students` 구조 확인


In [ ]:
sql = """
USE TODO;
SHOW TODO;
DESC TODO;
"""

_todo_id = 1
run_sql(sql)

In [ ]:
# TODO 1 자동점검 - 수정하지 마세요.
check_todo(1, sql, globals().get('_todo_id'))


# 5. SELECT 기본 문법

가장 기본적인 형태는 다음과 같습니다.

```sql
SELECT 컬럼명
FROM 테이블명;
```

쉽게 풀면:

```text
어떤 컬럼을 볼지 정하고,
어느 테이블에서 가져올지 정한다.
```

예:

```sql
SELECT name
FROM students;
```

이 SQL은 `students` 테이블에서 `name` 컬럼만 보여 달라는 뜻입니다.

---

## TODO 2 - 이름만 조회

`students` 테이블에서 `name` 컬럼만 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO
FROM TODO;
"""

_todo_id = 2
run_sql(sql)

In [ ]:
# TODO 2 자동점검 - 수정하지 마세요.
check_todo(2, sql, globals().get('_todo_id'))


# 6. 전체 데이터 조회하기

모든 컬럼을 보고 싶을 때는 `*`를 사용합니다.

```sql
SELECT * FROM students;
```

여기서 `*`는 **모든 컬럼**이라는 뜻입니다.

---

## TODO 3

`students` 테이블의 전체 데이터를 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO
FROM TODO;
"""

_todo_id = 3
run_sql(sql)

In [ ]:
# TODO 3 자동점검 - 수정하지 마세요.
check_todo(3, sql, globals().get('_todo_id'))


# 7. 원하는 컬럼만 조회하기

실제로는 항상 `SELECT *`를 쓰는 것보다 필요한 컬럼만 선택하는 것이 좋습니다.

예:

```sql
SELECT name, class_name
FROM students;
```

여러 컬럼은 쉼표(`,`)로 구분합니다.

---

## TODO 4

이름, 학년, 학과를 조회하세요.

필요한 컬럼:

```text
name
grade
class_name
```


In [ ]:
sql = """
USE school;

SELECT TODO, TODO, TODO
FROM students;
"""

_todo_id = 4
run_sql(sql)

In [ ]:
# TODO 4 자동점검 - 수정하지 마세요.
check_todo(4, sql, globals().get('_todo_id'))


# 8. 컬럼 이름에 별명 붙이기 — AS

조회 결과의 컬럼 이름을 보기 좋게 바꿀 수 있습니다.

예:

```sql
SELECT name AS 이름,
       grade AS 학년,
       class_name AS 학과
FROM students;
```

`AS`는 조회 결과에만 적용되는 **별명(alias)**입니다.  
실제 테이블의 컬럼 이름이 바뀌는 것은 아닙니다.

---

## TODO 5

다음 별명을 붙여 조회하세요.

- `name` → 이름
- `grade` → 학년
- `class_name` → 학과


In [ ]:
sql = """
USE school;

SELECT
    name AS TODO,
    grade AS TODO,
    class_name AS TODO
FROM students;
"""

_todo_id = 5
run_sql(sql)

In [ ]:
# TODO 5 자동점검 - 수정하지 마세요.
check_todo(5, sql, globals().get('_todo_id'))


# 9. 조건을 걸어 조회하기 — WHERE

모든 데이터가 아니라 필요한 데이터만 찾고 싶을 때 `WHERE`를 사용합니다.

예:

```sql
SELECT *
FROM students
WHERE grade = 3;
```

뜻:

```text
students 테이블에서
grade 값이 3인 데이터만
모든 컬럼으로 보여줘.
```

---

## TODO 6

3학년 학생만 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO = TODO;
"""

_todo_id = 6
run_sql(sql)

In [ ]:
# TODO 6 자동점검 - 수정하지 마세요.
check_todo(6, sql, globals().get('_todo_id'))


# 10. 문자 조건 조회하기

문자 데이터는 작은따옴표 `' '`로 감쌉니다.

예:

```sql
WHERE class_name = '컴퓨터과'
```

반면 숫자는 따옴표 없이 사용할 수 있습니다.

```sql
WHERE grade = 3
```

---

## TODO 7

`컴퓨터과` 학생만 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO = 'TODO';
"""

_todo_id = 7
run_sql(sql)

In [ ]:
# TODO 7 자동점검 - 수정하지 마세요.
check_todo(7, sql, globals().get('_todo_id'))


# 11. 비교 연산자 사용하기

`WHERE`에서는 다양한 비교 연산자를 사용할 수 있습니다.

| 연산자 | 의미 |
|---|---|
| `=` | 같다 |
| `!=` | 같지 않다 |
| `<>` | 같지 않다 |
| `>` | 크다 |
| `>=` | 크거나 같다 |
| `<` | 작다 |
| `<=` | 작거나 같다 |

예:

```sql
WHERE grade >= 2;
```

→ 2학년 이상

---

## TODO 8

2학년 이상인 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE grade TODO TODO;
"""

_todo_id = 8
run_sql(sql)

In [ ]:
# TODO 8 자동점검 - 수정하지 마세요.
check_todo(8, sql, globals().get('_todo_id'))


# 12. 같지 않다 조건

1학년이 아닌 학생을 조회하려면 다음 두 방식이 가능합니다.

```sql
WHERE grade != 1;
```

또는

```sql
WHERE grade <> 1;
```

둘 다 같은 의미입니다.

---

## TODO 9

1학년이 아닌 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE grade TODO TODO;
"""

_todo_id = 9
run_sql(sql)

In [ ]:
# TODO 9 자동점검 - 수정하지 마세요.
check_todo(9, sql, globals().get('_todo_id'))


# 13. 여러 조건 함께 사용하기 — AND

`AND`는 **모든 조건을 만족해야** 합니다.

예:

```sql
WHERE grade = 3
AND class_name = '컴퓨터과';
```

뜻:

> 3학년이면서 컴퓨터과인 학생

---

## TODO 10

3학년이면서 컴퓨터과인 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO = TODO
AND TODO = 'TODO';
"""

_todo_id = 10
run_sql(sql)

In [ ]:
# TODO 10 자동점검 - 수정하지 마세요.
check_todo(10, sql, globals().get('_todo_id'))


# 14. 여러 조건 함께 사용하기 — OR

`OR`는 조건 중 **하나만 만족해도** 됩니다.

예:

```sql
WHERE grade = 1
OR grade = 2;
```

뜻:

> 1학년이거나 2학년인 학생

---

## TODO 11

1학년 또는 2학년 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO = TODO
OR TODO = TODO;
"""

_todo_id = 11
run_sql(sql)

In [ ]:
# TODO 11 자동점검 - 수정하지 마세요.
check_todo(11, sql, globals().get('_todo_id'))


# 15. IN으로 여러 값 조회하기

OR 조건이 많아지면 SQL이 길어질 수 있습니다.

예:

```sql
WHERE grade = 1
OR grade = 2;
```

다음처럼 줄일 수 있습니다.

```sql
WHERE grade IN (1, 2);
```

뜻은 같습니다.

---

## TODO 12

`IN`을 이용해 1학년 또는 2학년 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO IN (TODO, TODO);
"""

_todo_id = 12
run_sql(sql)

In [ ]:
# TODO 12 자동점검 - 수정하지 마세요.
check_todo(12, sql, globals().get('_todo_id'))


# 16. 문자값도 IN으로 조회할 수 있다

`IN` 안에는 문자값도 사용할 수 있습니다.

예:

```sql
WHERE class_name IN ('컴퓨터과', '서버반');
```

---

## TODO 13

`컴퓨터과` 또는 `서버반` 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO IN ('TODO', 'TODO');
"""

_todo_id = 13
run_sql(sql)

In [ ]:
# TODO 13 자동점검 - 수정하지 마세요.
check_todo(13, sql, globals().get('_todo_id'))


# 17. LIKE로 포함된 글자 찾기

이름에 특정 글자가 들어간 데이터를 찾고 싶을 때 `LIKE`를 사용합니다.

`%`는 **아무 글자나 0개 이상 올 수 있음**을 의미합니다.

예:

```text
'김%'   → 김으로 시작
'%김'   → 김으로 끝남
'%김%'  → 김이 포함됨
```

---

## TODO 14

이름이 `김`으로 시작하는 학생을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE name LIKE 'TODO';
"""

_todo_id = 14
run_sql(sql)

In [ ]:
# TODO 14 자동점검 - 수정하지 마세요.
check_todo(14, sql, globals().get('_todo_id'))


# 18. ORDER BY로 정렬하기

정렬할 때는 `ORDER BY`를 사용합니다.

오름차순:

```sql
ORDER BY grade ASC;
```

내림차순:

```sql
ORDER BY grade DESC;
```

### ASC
작은 값 → 큰 값

### DESC
큰 값 → 작은 값

---

## TODO 15

학년을 오름차순으로 정렬하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
ORDER BY TODO TODO;
"""

_todo_id = 15
run_sql(sql)

In [ ]:
# TODO 15 자동점검 - 수정하지 마세요.
check_todo(15, sql, globals().get('_todo_id'))


# 19. 내림차순 정렬

## TODO 16

학년이 높은 학생부터 나오도록 정렬하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
ORDER BY TODO TODO;
"""

_todo_id = 16
run_sql(sql)

In [ ]:
# TODO 16 자동점검 - 수정하지 마세요.
check_todo(16, sql, globals().get('_todo_id'))


# 20. 여러 기준으로 정렬하기

정렬 기준을 여러 개 줄 수도 있습니다.

예:

```sql
ORDER BY grade ASC, name ASC;
```

뜻:

1. 먼저 학년을 오름차순으로 정렬
2. 같은 학년 안에서는 이름을 오름차순으로 정렬

---

## TODO 17

학년 오름차순, 같은 학년에서는 이름 오름차순으로 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
ORDER BY TODO TODO, TODO TODO;
"""

_todo_id = 17
run_sql(sql)

In [ ]:
# TODO 17 자동점검 - 수정하지 마세요.
check_todo(17, sql, globals().get('_todo_id'))


# 21. LIMIT으로 조회 개수 제한하기

데이터가 많을 때 일부만 보고 싶다면 `LIMIT`을 사용합니다.

예:

```sql
SELECT * FROM students
LIMIT 3;
```

→ 앞의 3개 행만 조회

---

## TODO 18

학생 데이터 3개만 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
LIMIT TODO;
"""

_todo_id = 18
run_sql(sql)

In [ ]:
# TODO 18 자동점검 - 수정하지 마세요.
check_todo(18, sql, globals().get('_todo_id'))


# 22. 최근 데이터 3개 보기

`ORDER BY`와 `LIMIT`을 함께 쓰면 더 유용합니다.

```sql
ORDER BY id DESC
LIMIT 3;
```

뜻:

1. `id`를 큰 값부터 정렬
2. 그중 앞의 3개만 표시

즉, 최근 입력된 데이터 3개를 확인할 수 있습니다.

---

## TODO 19

최근 입력된 학생 3명을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
ORDER BY TODO TODO
LIMIT TODO;
"""

_todo_id = 19
run_sql(sql)

In [ ]:
# TODO 19 자동점검 - 수정하지 마세요.
check_todo(19, sql, globals().get('_todo_id'))


# 23. DISTINCT로 중복 제거하기

학과 목록만 조회하면 같은 학과가 여러 번 나올 수 있습니다.

```sql
SELECT class_name
FROM students;
```

중복을 제거하려면:

```sql
SELECT DISTINCT class_name
FROM students;
```

`DISTINCT`는 같은 값을 한 번만 보여 줍니다.

---

## TODO 20

중복 없는 학과 목록을 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO TODO
FROM students;
"""

_todo_id = 20
run_sql(sql)

In [ ]:
# TODO 20 자동점검 - 수정하지 마세요.
check_todo(20, sql, globals().get('_todo_id'))


# 24. COUNT(*)로 데이터 개수 세기

테이블 안에 데이터가 몇 줄 있는지 확인할 때:

```sql
SELECT COUNT(*)
FROM students;
```

를 사용합니다.

`COUNT(*)`은 전체 행의 개수를 셉니다.

---

## TODO 21

students 테이블의 전체 행 개수를 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO
FROM students;
"""

_todo_id = 21
run_sql(sql)

In [ ]:
# TODO 21 자동점검 - 수정하지 마세요.
check_todo(21, sql, globals().get('_todo_id'))


# 25. 조건과 COUNT(*) 함께 사용하기

`WHERE`와 함께 사용하면 조건에 맞는 행의 개수만 셀 수 있습니다.

예:

```sql
SELECT COUNT(*)
FROM students
WHERE grade = 1;
```

→ 1학년 학생 수

---

## TODO 22

1학년 학생 수를 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO
FROM students
WHERE TODO = TODO;
"""

_todo_id = 22
run_sql(sql)

In [ ]:
# TODO 22 자동점검 - 수정하지 마세요.
check_todo(22, sql, globals().get('_todo_id'))


# 26. 종합 문제 1

다음 조건을 만족하는 SQL을 작성하세요.

### 조건

- `school` 데이터베이스 사용
- 2학년 이상 학생
- 이름과 학년만 출력
- 학년은 높은 순서
- 같은 학년에서는 이름 오름차순

## TODO 23


In [ ]:
sql = """





"""

_todo_id = 23
run_sql(sql)

In [ ]:
# TODO 23 자동점검 - 수정하지 마세요.
check_todo(23, sql, globals().get('_todo_id'))


# 27. 종합 문제 2

다음 조건을 만족하는 SQL을 작성하세요.

### 조건

- 학과가 `컴퓨터과` 또는 `서버반`
- 이름, 학년, 학과를 출력
- 이름 오름차순

## TODO 24


In [ ]:
sql = """





"""

_todo_id = 24
run_sql(sql)

In [ ]:
# TODO 24 자동점검 - 수정하지 마세요.
check_todo(24, sql, globals().get('_todo_id'))


# 28. 종합 문제 3

다음 조건을 만족하는 SQL을 작성하세요.

### 조건

- 이름에 `김`이 포함된 학생
- 전체 컬럼 조회

힌트:

```text
'%김%'
```

## TODO 25


In [ ]:
sql = """



"""

_todo_id = 25
run_sql(sql)

In [ ]:
# TODO 25 자동점검 - 수정하지 마세요.
check_todo(25, sql, globals().get('_todo_id'))


# 29. 자주 발생하는 오류

## ① 세미콜론 누락

잘못된 예:

```sql
SELECT * FROM students
```

올바른 예:

```sql
SELECT * FROM students;
```

---

## ② 데이터베이스를 선택하지 않음

오류 예:

```text
No database selected
```

해결:

```sql
USE school;
```

---

## ③ 테이블 이름 오타

잘못된 예:

```sql
SELECT * FROM student;
```

올바른 예:

```sql
SELECT * FROM students;
```

확인:

```sql
SHOW TABLES;
```

---

## ④ 문자에 따옴표를 쓰지 않음

잘못된 예:

```sql
WHERE name = 김부장;
```

올바른 예:

```sql
WHERE name = '김부장';
```


# 30. 전체 TODO 자동점검 결과

각 TODO 아래의 자동점검 셀을 실행하면 결과가 저장됩니다.

- ✅ : 해당 TODO의 실행 결과가 정확함
- ❌ : 실행했지만 결과가 다르거나 SQL 오류가 있음
- ⬜ : 아직 자동점검 셀을 실행하지 않음

아래 셀에서 **TODO 1~25 전체 진행률**을 확인하세요.

> 주의: Colab에서 런타임을 다시 시작하면 점검 기록도 초기화됩니다.  
> 그 경우 위에서부터 셀을 다시 실행하세요.


In [ ]:
show_todo_progress(25)

print("\n===== 데이터베이스 기본 상태 확인 =====")

def mysql_value(query):
    result = subprocess.run(
        ["sudo", "mysql", "-N", "-B", "-e", query],
        text=True,
        capture_output=True
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

checks = {
    "전체 학생 7명": ("SELECT COUNT(*) FROM school.students;", "7"),
    "3학년 2명": ("SELECT COUNT(*) FROM school.students WHERE grade=3;", "2"),
    "1학년 3명": ("SELECT COUNT(*) FROM school.students WHERE grade=1;", "3"),
    "컴퓨터과 2명": ("SELECT COUNT(*) FROM school.students WHERE class_name='컴퓨터과';", "2"),
    "김으로 시작 2명": ("SELECT COUNT(*) FROM school.students WHERE name LIKE '김%';", "2"),
}

for label, (query, expected) in checks.items():
    value = mysql_value(query)
    mark = "✅" if value == expected else f"❌ 현재 {value}"
    print(f"{label:<16} : {mark}")



# 31. 오늘 배운 내용 정리

핵심 SQL:

```sql
SELECT
FROM
WHERE
AND
OR
IN
LIKE
ORDER BY
ASC
DESC
LIMIT
DISTINCT
COUNT(*)
AS
```

핵심 의미:

```text
SELECT     → 데이터를 조회
*          → 모든 컬럼
AS         → 조회 결과 컬럼 이름에 별명
WHERE      → 조건 지정
AND        → 모든 조건 만족
OR         → 조건 중 하나 만족
IN         → 여러 값 중 하나와 일치
LIKE       → 특정 글자가 포함된 데이터 검색
ORDER BY   → 정렬
ASC        → 오름차순
DESC       → 내림차순
LIMIT      → 조회 개수 제한
DISTINCT   → 중복 제거
COUNT(*)   → 행 개수 계산
```

---

## 김부장식 정리

> **SELECT는 데이터베이스에게 묻는 질문입니다.**

> "어떤 테이블에서,  
> 어떤 항목을,  
> 어떤 조건으로 보여줄래?"



# 32. 자기 점검

- [ ] SELECT의 역할을 설명할 수 있다.
- [ ] 전체 데이터를 조회할 수 있다.
- [ ] 원하는 컬럼만 조회할 수 있다.
- [ ] AS를 사용할 수 있다.
- [ ] WHERE로 조건을 지정할 수 있다.
- [ ] 숫자 조건과 문자 조건의 차이를 알고 있다.
- [ ] 비교 연산자를 사용할 수 있다.
- [ ] AND와 OR의 차이를 설명할 수 있다.
- [ ] IN을 사용할 수 있다.
- [ ] LIKE와 `%`의 의미를 설명할 수 있다.
- [ ] ORDER BY로 정렬할 수 있다.
- [ ] ASC와 DESC의 차이를 설명할 수 있다.
- [ ] LIMIT으로 개수를 제한할 수 있다.
- [ ] DISTINCT로 중복을 제거할 수 있다.
- [ ] COUNT(*)로 데이터 개수를 셀 수 있다.



# 33. [선택] 실습 데이터 다시 초기화하기

실습 데이터를 변경했거나 처음부터 다시 하고 싶다면 아래 셀을 사용하세요.

`RESET = True`로 바꾸면 `students` 테이블의 데이터를 지우고 원래 7개 실습 데이터를 다시 입력합니다.


In [ ]:
RESET = False

if RESET:
    !sudo mysql -e "USE school; TRUNCATE TABLE students;"
    !sudo mysql -e "USE school; INSERT INTO students (name,grade,class_name) VALUES ('김부장',3,'컴퓨터과'),('혼이',1,'영혼반'),('라즈베리',2,'임베디드반'),('리눅스',3,'서버반'),('파이썬',1,'프로그래밍반'),('김코딩',1,'컴퓨터과'),('데이터왕',2,'데이터베이스반');"
    print("✅ 실습 데이터 7개를 다시 준비했습니다.")
else:
    print("초기화하지 않았습니다.")
